# 5.1.2 Dimension-Factor Correlation Analysis (Strict Statistics beta_3.00e-06)

This notebook performs a rigorous correlation analysis between VAE latent dimensions and clinical factors.
It implements strict filtering and statistical correction to identify robust signals.

## Methodology
1. **KL Filtering (Active Units)**: Only dimensions with Mean KL Divergence > Threshold (or High Variance) are analyzed.
2. **Strict Significance**: Bonferroni correction is applied to control the Family-Wise Error Rate (FWER).
   - $\alpha_{corrected} = 0.05 / N_{active}$
3. **Ranking & Separation**: The top dimension matches must pass the corrected significance threshold and are evaluated on separation quality (Gap %).

## Factors Analyzed
- **Age** (Continuous) -> Pearson Correlation
- **Sex** (Binary) -> Point-Biserial Correlation
- **SBR (PC1)** (Continuous) -> Pearson Correlation
- **Scanner** (Categorical) -> Eta-Squared (${\eta}^2$)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pointbiserialr, f_oneway
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import os
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4e' % x)

## 1. Parameterization & Data Loading

In [2]:
# --- PARAMETERS ---
TARGET_BETA = "3.00e-06"
KL_THRESHOLD = 0.05
ALPHA_LEVEL = 0.05

# Paths
LATENT_CSV = "output/final_train_combined_vae_data_beta_3.00e-06.csv"
# Optional: Path to a pre-computed KL divergence CSV. 
# If file not found, we use Latent Variance as a proxy for activity.
KL_CSV_PATH = f"output/Experiments/BetaScanVAE/beta_scan_results/beta_{TARGET_BETA}/kl_divergence.csv"

print(f"Target Beta Model: {TARGET_BETA}")
print(f"KL Threshold for Active Units: {KL_THRESHOLD}")
print(f"Initial Significance Level: {ALPHA_LEVEL}")

Target Beta Model: 3.00e-06
KL Threshold for Active Units: 0.05
Initial Significance Level: 0.05


In [3]:
# Load Main Data
if os.path.exists(LATENT_CSV):
    df_latent = pd.read_csv(LATENT_CSV)
    print(f"Loaded Latent Data: {df_latent.shape}")
else:
    raise FileNotFoundError(f"Latent data not found at {LATENT_CSV}. Please run 5.0_prepare_csv_data_vae.ipynb first.")

# Define Latent Columns (latent_0 ... latent_255)
DIM_COLS = [c for c in df_latent.columns if c.startswith('latent_')]
print(f"Total Latent Dimensions: {len(DIM_COLS)}")

Loaded Latent Data: (2373, 303)
Total Latent Dimensions: 256


In [4]:
SBR_COLS = [
    'DATSCAN_CAUDATE_R', 'DATSCAN_CAUDATE_L',
    'DATSCAN_PUTAMEN_R', 'DATSCAN_PUTAMEN_L',
    'DATSCAN_PUTAMEN_R_ANT', 'DATSCAN_PUTAMEN_L_ANT'
]

RANDOM_STATE = 42

# Filter to rows with complete SBR data
df_with_sbr = df_latent.dropna(subset=SBR_COLS).copy()
print(f"Rows with complete SBR data: {len(df_with_sbr)}")

# Standardize and apply PCA
scaler_sbr = StandardScaler()
sbr_scaled = scaler_sbr.fit_transform(df_with_sbr[SBR_COLS])
pca = PCA(n_components=3, random_state=RANDOM_STATE)
sbr_pcs = pca.fit_transform(sbr_scaled)

df_with_sbr['SBR_PC1'] = sbr_pcs[:, 0]
df_with_sbr['SBR_PC2'] = sbr_pcs[:, 1]
df_with_sbr['SBR_PC3'] = sbr_pcs[:, 2]

print(f"\nSBR PCA explained variance: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.2%}")

# Use this as final dataset
df_final = df_with_sbr.reset_index(drop=True)
print(f"\nFinal dataset: {df_final.shape}")
print(f"Unique patients: {df_final['PATNO'].nunique()}")

Rows with complete SBR data: 2373

SBR PCA explained variance: [0.88338287 0.05277345 0.04814441]
Total variance explained: 98.43%

Final dataset: (2373, 306)
Unique patients: 1437


## 2. Active Unit Filtering (Request #1)
We filter out "collapsed" dimensions (posterior collapse). Valid dimensions must have significant information content.

In [5]:
active_features = []
kl_values = {}

# Check if specific KL file exists
if os.path.exists(KL_CSV_PATH):
    print(f"Loading KL Divergence from: {KL_CSV_PATH}")
    df_kl = pd.read_csv(KL_CSV_PATH)
    # Assuming format: dimension, mean_kl
    for _, row in df_kl.iterrows():
        if row['mean_kl'] > KL_THRESHOLD:
            active_features.append(row['dimension'])
            kl_values[row['dimension']] = row['mean_kl']
else:
    print("Warning: KL Divergence file not found. Using Latent Variance as proxy for activity.")
    print(f"(Threshold > {KL_THRESHOLD})")
    
    variances = df_latent[DIM_COLS].var()
    for dim, var in variances.items():
        # In VAEs, collapsed units often have variance close to 0 (or close to prior)
        # Here we assume active units have variance significantly > threshold
        if var > KL_THRESHOLD:
            active_features.append(dim)
            kl_values[dim] = var # Storing variance as proxy metric

ACTIVE_FEATURES = sorted(active_features, key=lambda x: int(x.split('_')[1]))
N_ACTIVE = len(ACTIVE_FEATURES)

print(f"\nActive Dimensions identified: {N_ACTIVE} / {len(DIM_COLS)}")
print(f"Collapsed Dimensions: {len(DIM_COLS) - N_ACTIVE}")

(Threshold > 0.05)

Active Dimensions identified: 240 / 256
Collapsed Dimensions: 16


## 3. Statistical Correction (Request #2)
Calculating Bonferroni-corrected significance threshold.

In [6]:
if N_ACTIVE > 0:
    ALPHA_CORRECTED = ALPHA_LEVEL / N_ACTIVE
else:
    ALPHA_CORRECTED = ALPHA_LEVEL # Fallback to avoid division by zero if all collapsed
    
print(f"Bonferroni Corrected Alpha: {ALPHA_CORRECTED:.2e}")
print(f"(Base Alpha {ALPHA_LEVEL} / {N_ACTIVE} Tests)")

Bonferroni Corrected Alpha: 2.08e-04
(Base Alpha 0.05 / 240 Tests)


## 4. Correlation Analysis Modules (Request #4)
Functions to compute correlations and rank dimensions.

In [7]:
def analyze_factor(data, factor_col, factor_type, active_dims, alpha_threshold):
    results = []
    
    # Remove NaNs for this factor
    valid_data = data.dropna(subset=[factor_col])
    if len(valid_data) < 5:
        print(f"Not enough data for {factor_col}")
        return pd.DataFrame()
    
    y = valid_data[factor_col]
    
    for dim in active_dims:
        x = valid_data[dim]
        
        stat = 0
        p_val = 1.0
        effect_size = 0
        
        if factor_type == 'continuous':
            # Pearson Correlation
            r, p = stats.pearsonr(x, y)
            stat = r
            p_val = p
            effect_size = abs(r) # Absolute correlation strength
            
        elif factor_type == 'binary':
            # Point Biserial
            # Ensure binary is numeric
            r, p = pointbiserialr(y, x)
            stat = r
            p_val = p
            effect_size = abs(r)
            
        elif factor_type == 'categorical':
            # Eta-Squared (from ANOVA)
            groups = [valid_data[valid_data[factor_col] == g][dim] for g in y.unique()]
            if len(groups) > 1:
                f_stat, p = f_oneway(*groups)
                # Calculate Eta-Squared: SS_between / SS_total
                # Simplified from F-stat: eta2 = (F * (k-1)) / (F * (k-1) + (N-k))
                k = len(groups)
                N = len(valid_data)
                eta2 = (f_stat * (k - 1)) / (f_stat * (k - 1) + (N - k))
                
                stat = f_stat
                p_val = p
                effect_size = eta2
            else:
                continue
                
        results.append({
            'Dimension': dim,
            'Statistic': stat,
            'Effect_Size': effect_size,
            'P_Value': p_val,
            'Survives_Correction': p_val < alpha_threshold
        })
        
    res_df = pd.DataFrame(results)
    if not res_df.empty:
        res_df = res_df.sort_values(by='Effect_Size', ascending=False).reset_index(drop=True)
        res_df['Rank'] = res_df.index + 1
        
    return res_df

def interpret_gap(gap_pct):
    if gap_pct >= 20: return "Excellent"
    if gap_pct >= 10: return "Good"
    if gap_pct >= 5:  return "Moderate"
    return "Weak"

## 5. Execution & Summary Generation

In [8]:
factors_config = [
    {'name': 'Age', 'col': 'AGE_AT_VISIT', 'type': 'continuous'},
    {'name': 'Sex', 'col': 'SEX', 'type': 'binary'},
    {'name': 'Scanner', 'col': 'Manufacturer', 'type': 'categorical'},
    {'name': 'Scanner Model', 'col': 'ManufacturerModelName', 'type': 'categorical'},
    {'name': 'Patient Hand', 'col': 'HANDED', 'type': 'categorical'},
    {'name': 'SBR_PC1', 'col': 'SBR_PC1', 'type': 'continuous'},
    {'name': 'SBR_PC2', 'col': 'SBR_PC2', 'type': 'continuous'},
    {'name': 'SBR_PC3', 'col': 'SBR_PC3', 'type': 'continuous'}
]

summary_rows = []

for factor in factors_config:
    print(f"\nAnalyzing Factor: {factor['name']}...")
    df_res = analyze_factor(df_final, factor['col'], factor['type'], ACTIVE_FEATURES, ALPHA_CORRECTED)
    
    if df_res.empty:
        print("No results.")
        continue
        
    # Get Top 2 for Gap Analysis
    top_1 = df_res.iloc[0]
    top_2 = df_res.iloc[1] if len(df_res) > 1 else None
    
    gap_abs = 0.0
    gap_pct = 0.0
    sep_quality = "N/A"
    
    if top_2 is not None:
        gap_abs = top_1['Effect_Size'] - top_2['Effect_Size']
        if top_1['Effect_Size'] > 0:
            gap_pct = (gap_abs / top_1['Effect_Size']) * 100
        sep_quality = interpret_gap(gap_pct)
        
    print(f"  Top Dimension: {top_1['Dimension']}")
    print(f"  Effect Size: {top_1['Effect_Size']:.4f}")
    print(f"  P-Value: {top_1['P_Value']:.4e} (Corrected Threshold: {ALPHA_CORRECTED:.2e})")
    print(f"  Survives Correction: {top_1['Survives_Correction']}")
    print(f"  Separation Gap: {gap_pct:.1f}% ({sep_quality})")
    
    # Add to summary if it survives correction
    if top_1['Survives_Correction']:
        summary_rows.append({
            'Factor': factor['name'],
            'Top_Dimension': top_1['Dimension'],
            'Effect_Size': top_1['Effect_Size'],
            'P_Value': top_1['P_Value'],
            'Survives_Bonferroni': True,
            'Separation_Gap_Pct': gap_pct,
            'Separation_Quality': sep_quality
        })
    else:
        # Optional: Add but mark as non-significant
        summary_rows.append({
            'Factor': factor['name'],
            'Top_Dimension': top_1['Dimension'],
            'Effect_Size': top_1['Effect_Size'],
            'P_Value': top_1['P_Value'],
            'Survives_Bonferroni': False,
            'Separation_Gap_Pct': gap_pct,
            'Separation_Quality': "Insignificant"
        })

df_summary = pd.DataFrame(summary_rows)

print("\n" + "="*60)
print(f"FINAL SUMMARY TABLE (Strict Statistics) ({TARGET_BETA})")
print("="*60)
if not df_summary.empty:
    print(df_summary[['Factor', 'Top_Dimension', 'Effect_Size', 'P_Value', 'Survives_Bonferroni', 'Separation_Quality']].to_string(index=False))
else:
    print("No factors produced significant dimensions passing strict statistical correction.")


Analyzing Factor: Age...
  Top Dimension: latent_72
  Effect Size: 0.3407
  P-Value: 1.5801e-65 (Corrected Threshold: 2.08e-04)
  Survives Correction: True
  Separation Gap: 16.0% (Good)

Analyzing Factor: Sex...
  Top Dimension: latent_133
  Effect Size: 0.2278
  P-Value: 2.7187e-29 (Corrected Threshold: 2.08e-04)
  Survives Correction: True
  Separation Gap: 8.5% (Moderate)

Analyzing Factor: Scanner...
  Top Dimension: latent_213
  Effect Size: 0.1707
  P-Value: 8.5855e-85 (Corrected Threshold: 2.08e-04)
  Survives Correction: True
  Separation Gap: 12.2% (Good)

Analyzing Factor: Scanner Model...
  Top Dimension: latent_213
  Effect Size: 0.1861
  P-Value: 7.3196e-87 (Corrected Threshold: 2.08e-04)
  Survives Correction: True
  Separation Gap: 2.5% (Weak)

Analyzing Factor: Patient Hand...
  Top Dimension: latent_23
  Effect Size: 0.0050
  P-Value: 2.7511e-03 (Corrected Threshold: 2.08e-04)
  Survives Correction: False
  Separation Gap: 5.4% (Moderate)

Analyzing Factor: SBR_PC1..

In [9]:
# Optional: Save Results
if not df_summary.empty:
    output_path = f"output/strict_correlation_analysis_summary_{TARGET_BETA}.csv"
    df_summary.to_csv(output_path, index=False)
    print(f"\nSummary saved to: {output_path}")


Summary saved to: output/strict_correlation_analysis_summary_3.00e-06.csv
